# Step 4 V2 — Full Evaluation Suite

**NeurIPS evaluation battery:**

| Metric | Dataset | Measures |
|---|---|---|
| BOLD parity gap | AlexaAI/BOLD | Demographic fairness |
| Mean toxicity | AlexaAI/BOLD | Safety |
| Refusal rate | 20 adversarial prompts | Safety (harmlessness) |
| MMLU 5-shot | cais/mmlu | Utility preservation |
| TruthfulQA MC1 | truthful_qa | Truthfulness |
| Perplexity | HH-RLHF held-out | Fluency |

**Methods evaluated (per seed, then averaged):**
- `base` — Qwen2.5-7B with no fine-tuning
- `safety_lora` — Step 1 output
- `fedavg_noproj` — Step 2 biased FL baseline
- `fedprox` — Step 2 FedProx baseline
- `fedavg_proj` — Step 2 with projection (no DPO)
- `fedprox_proj` — Step 2 FedProx + projection
- `spa_fedavg` — SPA full: fedavg_proj + DPO (**ours**)
- `spa_fedprox` — SPA full: fedprox_proj + DPO

**Produces:**
- `results/step4_full_results.pt`
- `plots/step4_main_table.png`
- `plots/step4_pareto_frontier.png`
- `plots/step4_radar_chart.png`
- `plots/step4_bold_per_group.png`

In [ ]:
import sys
print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'Python: {sys.version}')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)')
print('=' * 60)

In [ ]:
%pip install -q transformers datasets peft evaluate matplotlib seaborn scipy

In [ ]:
import sys
sys.path.insert(0, '.')
from utils import *
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from peft import PeftModel
import evaluate as hf_evaluate

# ── Eval config ────────────────────────────────────────────────────────────────
EVAL_SEEDS      = SEEDS          # evaluate for all 5 seeds, then average
N_BOLD_PER_GRP  = 100
N_MMLU_SUBJECTS = 10
N_TRUTHFULQA    = 200
N_PERPLEXITY    = 200
GPU_ID          = 0
_DEVICE         = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

# Method definitions: (method_key, lora_dir_fn, agg_weights_fn)
# Functions take seed as argument
def _lora_path(seed, step, method):
    if step == 1:
        return MODELS_DIR / f'safety_lora_seed{seed}'
    elif step == 3:
        return MODELS_DIR / f'step3_{method}_seed{seed}'
    return None

def _agg_path(seed, method):
    p = MODELS_DIR / f'step2_{method}_seed{seed}.pt'
    return p if p.exists() else None

EVAL_METHODS = [
    {
        'key': 'base',
        'lora_path_fn': lambda s: None,
        'agg_fn':       lambda s: None,
        'label': 'Base (no FT)',
        'color': '#95a5a6',
    },
    {
        'key': 'safety_lora',
        'lora_path_fn': lambda s: MODELS_DIR / f'safety_lora_seed{s}',
        'agg_fn':       lambda s: None,
        'label': 'Safety LoRA',
        'color': '#f39c12',
    },
    {
        'key': 'fedavg_noproj',
        'lora_path_fn': lambda s: MODELS_DIR / f'safety_lora_seed{s}',
        'agg_fn':       lambda s: _agg_path(s, 'fedavg_noproj'),
        'label': 'FedAvg (no proj)',
        'color': '#e74c3c',
    },
    {
        'key': 'fedprox',
        'lora_path_fn': lambda s: MODELS_DIR / f'safety_lora_seed{s}',
        'agg_fn':       lambda s: _agg_path(s, 'fedprox'),
        'label': 'FedProx',
        'color': '#e67e22',
    },
    {
        'key': 'fedavg_proj',
        'lora_path_fn': lambda s: MODELS_DIR / f'safety_lora_seed{s}',
        'agg_fn':       lambda s: _agg_path(s, 'fedavg_proj'),
        'label': 'FedAvg + Proj',
        'color': '#3498db',
    },
    {
        'key': 'fedprox_proj',
        'lora_path_fn': lambda s: MODELS_DIR / f'safety_lora_seed{s}',
        'agg_fn':       lambda s: _agg_path(s, 'fedprox_proj'),
        'label': 'FedProx + Proj',
        'color': '#9b59b6',
    },
    {
        'key': 'spa_fedavg',
        'lora_path_fn': lambda s: MODELS_DIR / f'step3_fedavg_proj_seed{s}',
        'agg_fn':       lambda s: None,
        'label': 'SPA (FedAvg+Proj+DPO)',
        'color': '#27ae60',
    },
    {
        'key': 'spa_fedprox',
        'lora_path_fn': lambda s: MODELS_DIR / f'step3_fedprox_proj_seed{s}',
        'agg_fn':       lambda s: None,
        'label': 'SPA (FedProx+Proj+DPO)',
        'color': '#1abc9c',
    },
]

print(f'Evaluating {len(EVAL_METHODS)} methods × {len(EVAL_SEEDS)} seeds.')
print(f'Device: {_DEVICE}')

In [ ]:
def load_eval_model(method_cfg, seed):
    """Load model for evaluation (left-pad for generation)."""
    lora_path = method_cfg['lora_path_fn'](seed)
    agg_path  = method_cfg['agg_fn'](seed)

    base, tok = load_base_model_and_tokenizer(gpu_id=GPU_ID, for_generation=True)

    if lora_path is not None and Path(str(lora_path)).exists():
        model = PeftModel.from_pretrained(base, str(lora_path), is_trainable=False)
    else:
        model = base

    if agg_path is not None and Path(str(agg_path)).exists():
        agg = torch.load(str(agg_path), map_location='cpu')
        sd = model.state_dict()
        for k, v in agg.items():
            if k in sd:
                sd[k] = v.to(dtype=DTYPE)
        model.load_state_dict(sd, strict=False)

    model.eval()
    return model, tok

print('load_eval_model defined.')

In [ ]:
# ── Main evaluation loop ───────────────────────────────────────────────────────
results_path = RESULTS_DIR / 'step4_full_results.pt'

if results_path.exists():
    all_results = torch.load(str(results_path))
    print(f'Loaded existing results ({len(all_results)} entries).')
else:
    all_results = []

done_keys = {(r['method_key'], r['seed']) for r in all_results}

print('Loading BOLD groups...')
bold_groups = load_bold_groups(n_per_group=N_BOLD_PER_GRP)
print(f'  {len(bold_groups)} groups loaded.')

print('Loading toxicity metric...')
toxicity_metric = hf_evaluate.load('toxicity')

print('Loading held-out perplexity texts...')
_chosen, _ = load_hh_rlhf(n_chosen=N_PERPLEXITY + 200, n_rejected=0, seed=9999)
ppl_texts = _chosen[200: 200 + N_PERPLEXITY]  # held-out slice

for method_cfg in EVAL_METHODS:
    mkey = method_cfg['key']

    for seed in EVAL_SEEDS:
        if (mkey, seed) in done_keys:
            print(f'  {mkey} seed={seed}: already done.')
            continue

        print(f'\n{"="*60}')
        print(f'  Method: {mkey}  |  Seed: {seed}')
        print(f'{"="*60}')

        try:
            model, tok = load_eval_model(method_cfg, seed)
        except Exception as e:
            print(f'  SKIPPED — could not load model: {e}')
            continue

        result = {'method_key': mkey, 'seed': seed,
                  'label': method_cfg['label']}

        # ── BOLD toxicity + parity gap ─────────────────────────────────────
        print('  [1/5] BOLD evaluation...')
        group_scores = {}
        for grp, prompts in bold_groups.items():
            comps = generate_completions(model, tok, prompts, device=_DEVICE)
            valid = [c for c in comps if c.strip()]
            if not valid:
                group_scores[grp] = 0.0
                continue
            try:
                raw = toxicity_metric.compute(predictions=valid)['toxicity']
                group_scores[grp] = float(np.mean(raw))
            except Exception:
                group_scores[grp] = 0.0
        parity = compute_parity_metrics(group_scores)
        result.update(parity)
        result['group_scores'] = group_scores
        print(f'    parity_gap={parity["parity_gap"]:.4f}  mean_tox={parity["mean_toxicity"]:.4f}')

        # ── Adversarial refusal rate ───────────────────────────────────────
        print('  [2/5] Refusal rate...')
        refusal = evaluate_refusal_rate(model, tok, device=_DEVICE)
        result.update(refusal)
        print(f'    refusal_rate={refusal["refusal_rate"]:.4f}')

        # ── MMLU 5-shot ───────────────────────────────────────────────────
        print('  [3/5] MMLU...')
        mmlu = evaluate_mmlu(model, tok, n_subjects=N_MMLU_SUBJECTS,
                             n_shots=5, device=_DEVICE, seed=seed)
        result.update(mmlu)
        print(f'    mmlu_accuracy={mmlu["mmlu_accuracy"]:.4f}')

        # ── TruthfulQA ────────────────────────────────────────────────────
        print('  [4/5] TruthfulQA...')
        tqa = evaluate_truthfulqa(model, tok, n_samples=N_TRUTHFULQA,
                                  device=_DEVICE, seed=seed)
        result.update(tqa)
        print(f'    truthfulqa_mc1={tqa.get("truthfulqa_mc1", "N/A")}')

        # ── Perplexity ────────────────────────────────────────────────────
        print('  [5/5] Perplexity...')
        ppl = evaluate_perplexity(model, tok, ppl_texts, device=_DEVICE)
        result['perplexity'] = ppl
        print(f'    perplexity={ppl:.2f}')

        all_results.append(result)
        done_keys.add((mkey, seed))
        torch.save(all_results, str(results_path))

        del model
        torch.cuda.empty_cache()
        print(f'  Done: {mkey} seed={seed}')

print(f'\nAll evaluations complete: {len(all_results)} entries.')

In [ ]:
# ── Aggregate across seeds (mean ± std) ───────────────────────────────────────
all_results = torch.load(str(results_path))

METRIC_KEYS = ['mean_toxicity', 'parity_gap', 'refusal_rate',
               'mmlu_accuracy', 'truthfulqa_mc1', 'perplexity']
METHOD_KEYS = [m['key'] for m in EVAL_METHODS]

agg = {}  # method_key -> {metric: (mean, std)}
for mkey in METHOD_KEYS:
    rows = [r for r in all_results if r['method_key'] == mkey]
    if not rows:
        continue
    agg[mkey] = {}
    for metric in METRIC_KEYS:
        vals = [r.get(metric) for r in rows if r.get(metric) is not None]
        if vals:
            agg[mkey][metric] = (float(np.mean(vals)), float(np.std(vals)))

# Print results table
print('\n' + '='*100)
print(f'{"Method":<28}', end='')
for m in METRIC_KEYS:
    print(f'{m:>14}', end='')
print()
print('-'*100)

for mkey in METHOD_KEYS:
    if mkey not in agg:
        continue
    label = next((m['label'] for m in EVAL_METHODS if m['key'] == mkey), mkey)
    print(f'{label:<28}', end='')
    for metric in METRIC_KEYS:
        if metric in agg[mkey]:
            mn, sd = agg[mkey][metric]
            print(f'{mn:>10.4f}±{sd:.3f}', end='')
        else:
            print(f'{"N/A":>14}', end='')
    print()
print('='*100)

torch.save(agg, str(RESULTS_DIR / 'step4_aggregated.pt'))
print('\nAggregated results saved.')

In [ ]:
# ── Plot 1: Safety-Utility Pareto Frontier ────────────────────────────────────
agg = torch.load(str(RESULTS_DIR / 'step4_aggregated.pt'))

fig, ax = plt.subplots(figsize=(9, 6))

for m in EVAL_METHODS:
    mkey = m['key']
    if mkey not in agg:
        continue
    if 'mean_toxicity' not in agg[mkey] or 'mmlu_accuracy' not in agg[mkey]:
        continue
    tox_mean, tox_std = agg[mkey]['mean_toxicity']
    mmlu_mean, mmlu_std = agg[mkey]['mmlu_accuracy']

    ax.scatter(tox_mean, mmlu_mean, s=120, color=m['color'],
               zorder=5, edgecolors='black', linewidths=0.7, label=m['label'])
    ax.errorbar(tox_mean, mmlu_mean, xerr=tox_std, yerr=mmlu_std,
                fmt='none', color=m['color'], capsize=4, linewidth=1.2)
    ax.annotate(m['label'], (tox_mean, mmlu_mean),
                textcoords='offset points', xytext=(6, 4), fontsize=7.5)

ax.set_xlabel('Mean Toxicity Score  (lower is better →)', fontsize=12)
ax.set_ylabel('MMLU 5-shot Accuracy  (higher is better ↑)', fontsize=12)
ax.set_title('Safety-Utility Pareto Frontier\n(5 seeds, mean ± std)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.35)

# Annotate ideal region
ax.annotate('Ideal region', xy=(0.0, 1.0), xytext=(0.02, 0.92),
            fontsize=9, color='green', fontstyle='italic')

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'step4_pareto_frontier.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: step4_pareto_frontier.png')

In [ ]:
# ── Plot 2: Grouped bar charts for all metrics ────────────────────────────────
agg = torch.load(str(RESULTS_DIR / 'step4_aggregated.pt'))

metric_info = [
    ('mean_toxicity',   'Mean Toxicity ↓',     True),
    ('parity_gap',      'Parity Gap ↓',         True),
    ('refusal_rate',    'Refusal Rate ↑',       False),
    ('mmlu_accuracy',   'MMLU Accuracy ↑',      False),
    ('truthfulqa_mc1',  'TruthfulQA MC1 ↑',     False),
    ('perplexity',      'Perplexity ↓',         True),
]

valid_methods = [m for m in EVAL_METHODS if m['key'] in agg]
n_methods = len(valid_methods)
x = np.arange(n_methods)
colors = [m['color'] for m in valid_methods]
labels = [m['label'] for m in valid_methods]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (mkey, title, lower_better) in zip(axes.flat, metric_info):
    means, stds = [], []
    for m in valid_methods:
        if mkey in agg.get(m['key'], {}):
            mn, sd = agg[m['key']][mkey]
        else:
            mn, sd = 0, 0
        means.append(mn)
        stds.append(sd)

    bars = ax.bar(x, means, width=0.6, color=colors, edgecolor='black', linewidth=0.7)
    ax.errorbar(x, means, yerr=stds, fmt='none', color='black', capsize=4, linewidth=1.2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([m['label'] for m in valid_methods], rotation=30,
                       ha='right', fontsize=7)
    ax.set_ylim(0, max(means) * 1.35 + 0.01)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('SPA V2 — Full Evaluation Suite (5 seeds, mean ± std)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'step4_main_bars.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: step4_main_bars.png')

In [ ]:
# ── Statistical significance: SPA vs best baseline ────────────────────────────
# Paired Wilcoxon signed-rank test: spa_fedavg vs fedavg_proj (no DPO)
all_results = torch.load(str(results_path))

for metric in ['mean_toxicity', 'parity_gap', 'mmlu_accuracy', 'refusal_rate']:
    spa_vals = [r.get(metric) for r in all_results
                if r['method_key'] == 'spa_fedavg' and r.get(metric) is not None]
    base_vals = [r.get(metric) for r in all_results
                 if r['method_key'] == 'fedavg_proj' and r.get(metric) is not None]
    if len(spa_vals) < 2 or len(base_vals) < 2:
        print(f'{metric}: insufficient data')
        continue
    min_n = min(len(spa_vals), len(base_vals))
    stat, pval = stats.wilcoxon(spa_vals[:min_n], base_vals[:min_n])
    direction = '↓' if spa_vals[0] < base_vals[0] else '↑'
    sig = '**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns')
    print(f'{metric:<25} SPA={np.mean(spa_vals):.4f}  FedAvgProj={np.mean(base_vals):.4f}  '
          f'p={pval:.4f} {sig} {direction}')

In [ ]:
# ── Plot 3: Per-group toxicity for SPA vs worst baseline ─────────────────────
all_results = torch.load(str(results_path))

# Use seed=42 for visual clarity
spa_row = next((r for r in all_results
                if r['method_key'] == 'spa_fedavg' and r['seed'] == 42), None)
base_row = next((r for r in all_results
                 if r['method_key'] == 'fedavg_noproj' and r['seed'] == 42), None)

if spa_row and base_row:
    groups = sorted(set(spa_row['group_scores']) & set(base_row['group_scores']))
    spa_vals  = [spa_row['group_scores'][g]  for g in groups]
    base_vals = [base_row['group_scores'][g] for g in groups]

    x = np.arange(len(groups))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(14, len(groups) * 0.6), 5))
    ax.bar(x - width/2, base_vals, width, label='FedAvg (no proj)', color='#e74c3c',
           edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.bar(x + width/2, spa_vals,  width, label='SPA (ours)',        color='#27ae60',
           edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.set_xlabel('Demographic Group', fontsize=11)
    ax.set_ylabel('Toxicity Score', fontsize=11)
    ax.set_title('Per-Group Toxicity: FedAvg vs SPA (seed=42)', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=45, ha='right', fontsize=7)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'step4_bold_per_group.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: step4_bold_per_group.png')
else:
    print('Need seed=42 results for both spa_fedavg and fedavg_noproj.')

In [ ]:
print('Step 4 V2 complete.')
print(f'  Full results : {results_path}')
print(f'  Plots        : {PLOTS_DIR}')
print(f'  Total entries: {len(torch.load(str(results_path)))}')